In [3]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models  
from torchvision.datasets.folder import default_loader
import matplotlib.pyplot as plt
from PIL import Image

In [5]:
# 图片抽取与加载
cats_path = './data/validation/cats'
dogs_path = './data/validation/dogs'
cats_images = [os.path.join(cats_path, img) for img in os.listdir(cats_path)[:5]]
dogs_images = [os.path.join(dogs_path, img) for img in os.listdir(dogs_path)[:5]]
selected_images = cats_images + dogs_images
real_labels = ['Cat'] * 5 + ['Dog'] * 5

In [7]:
# 数据预处理
def preprocess_image(image_path, transform):
    image = default_loader(image_path)  # 使用默认加载器加载图片
    return transform(image)

transform = transforms.Compose([
    transforms.Resize((148, 148)),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])

In [9]:
# 加载模型
def load_model(model_path, model):
    model.load_state_dict(torch.load(model_path, map_location='cpu', weights_only=False))
    model.eval()
    return model

# LeNet5模型
class LeNet5(nn.Module):
    def __init__(self, num_classes=2):
        super(LeNet5, self).__init__()
        self.feature_extractor = nn.Sequential(
            nn.Conv2d(3, 6, kernel_size=5),  # 输入通道3，输出通道6，卷积核大小5
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2),  # 池化大小2，步长2
            nn.Conv2d(6, 16, kernel_size=5),  # 输入通道6，输出通道16，卷积核大小5
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2, stride=2)  # 池化大小2，步长2
        )
        # 手动计算特征图尺寸 (148-4)/2 -> (72-4)/2 = 34
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 34 * 34, 120),  # 16是通道数，34是特征图大小
            nn.ReLU(),
            nn.Linear(120, 84),
            nn.ReLU(),
            nn.Linear(84, num_classes)
        )

    def forward(self, x):
        x = self.feature_extractor(x)
        x = self.classifier(x)
        return x

In [11]:
models = {
    "LeNet5": load_model('./weights/lenet5_best_model.pth', LeNet5(num_classes=2)),
    "AlexNet": load_model('./weights/alexnet_best_model.pth', models.alexnet(num_classes=2)),
    "ResNet18": load_model('./weights/resnet18_best_model.pth', models.resnet18(num_classes=2))
}

In [12]:
# 绘图函数
def plot_predictions(images, real_labels, predictions, model_name, save_path):
    fig, axes = plt.subplots(2, 5, figsize=(20, 8))  # 总画布大小，垂直尺寸稍微加大
    for i, ax in enumerate(axes.flat):
        img = Image.open(images[i])
        img = img.resize((148, 148))  # 统一调整显示的图片大小
        ax.imshow(img)
        # 设置标题
        pred_label = predictions[i]
        real_label = real_labels[i]
        color = 'green' if pred_label == real_label else 'red'
        ax.set_title(f"Pred: {pred_label}\nReal: {real_label}", fontsize=16, color=color)
        ax.axis('off')
    # 设置总标题
    fig.suptitle(f"Predictions by {model_name}", fontsize=24)
    plt.tight_layout()
    plt.subplots_adjust(top=0.85, hspace=0.5)  # 为总标题留出空间，增加行间距
    plt.savefig(save_path)
    plt.show()
    plt.close()

In [13]:
os.makedirs('./Image', exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# 预测并绘图
for model_name, model in models.items():
    predictions = []
    for img_path in selected_images:
        img_tensor = preprocess_image(img_path, transform).unsqueeze(0).to(device)
        output = model(img_tensor)
        pred_idx = torch.argmax(output, dim=1).item()
        predictions.append('Cat' if pred_idx == 0 else 'Dog')
    save_path = f"./Image/{model_name}_predictions.png"
    plot_predictions(selected_images, real_labels, predictions, model_name, save_path)
    print(f"Predictions for {model_name} saved to {save_path}")